# Notebook 04 — Analysis and Plots
**Member: C** | Run after all experiment CSVs are in `results/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
token = 'ghp_xxxxxxxxxxxx'  # paste your GitHub token each session
repo_url = f'https://{token}@github.com/TIFFMUN/SC4001-Group133.git'

if not os.path.exists('/content/SC4001-Group133/.git'):
    os.system(f'git clone {repo_url} /content/SC4001-Group133')

os.chdir('/content/SC4001-Group133')
os.system(f'git pull {repo_url} main')
!pip install -r requirements.txt -q
!pip install -e . -q  # installs src/ as a package, fixes import issues on Colab Python 3.13

In [ ]:
# Redirect HF cache + data/ + results/ to Drive — run before any HF imports
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.evaluation.metrics import compute_summary_stats

sns.set_theme(style='whitegrid', palette='muted')
RESULTS_DIR = Path('results/')
PLOT_DIR = Path('results/plots/')
PLOT_DIR.mkdir(exist_ok=True)

def load_and_summarise(csv_path, exp_name):
    stats = compute_summary_stats(pd.read_csv(csv_path).to_dict(orient='records'))
    stats['experiment'] = exp_name
    return stats

# Add/remove rows as experiments complete
experiment_files = [
    ('exp0_zero_shot_gsm8k.csv',      'Exp 0: Zero-shot'),
    ('exp1_few_shot_gsm8k.csv',       'Exp 1: Few-shot CoT'),
    ('exp2_adaptation_r16_gsm8k.csv', 'Exp 2: Adaptation (r=16)'),
    ('exp3_distillation_gsm8k.csv',   'Exp 3: Distillation D1'),
    ('exp4_filtered_gsm8k.csv',       'Exp 4: Filtered D2'),
    ('exp5_d3_cross_task_arc.csv',    'Exp 5: Cross-task ARC'),
]

summaries = []
for fname, exp_name in experiment_files:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        summaries.append(load_and_summarise(str(fpath), exp_name))
    else:
        print(f'[SKIP] {fname} not found yet')

summary_df = pd.DataFrame(summaries)
summary_df

In [ ]:
# Plot 1: Accuracy comparison
fig, ax = plt.subplots(figsize=(10, 5))
bars = sns.barplot(data=summary_df, x='experiment', y='accuracy', ax=ax)
ax.set_title('Accuracy by Experiment', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
for bar, row in zip(bars.patches, summary_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{row.accuracy:.2%}', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot1_accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# Plot 2: Accuracy vs latency trade-off
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=summary_df, x='mean_latency_ms', y='accuracy', s=120, ax=ax)
for _, row in summary_df.iterrows():
    ax.annotate(row['experiment'], (row['mean_latency_ms'], row['accuracy']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_title('Accuracy vs Latency Trade-off', fontsize=13)
ax.set_xlabel('Mean Latency (ms/sample)')
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot2_accuracy_vs_latency.png', dpi=150)
plt.show()

In [ ]:
# Plot 3: A2 ablation — accuracy vs LoRA rank
# Fill in actual values after running rank sweep in notebook 02
a2_data = pd.DataFrame({
    'rank': [4, 8, 16, 32],
    'accuracy': [None, None, None, None],
    'mean_latency_ms': [None, None, None, None],
})

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
ax1.plot(a2_data['rank'], a2_data['accuracy'], marker='o', color='steelblue', label='Accuracy')
ax2.plot(a2_data['rank'], a2_data['mean_latency_ms'], marker='s', color='coral', linestyle='--', label='Latency')
ax1.set_xlabel('LoRA Rank (r)')
ax1.set_ylabel('Accuracy', color='steelblue')
ax2.set_ylabel('Mean Latency (ms)', color='coral')
ax1.set_title('A2 Ablation: Accuracy & Latency vs LoRA Rank')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot3_a2_rank_ablation.png', dpi=150)
plt.show()

In [ ]:
# Plot 4: A3 ablation — learning curve
# Fill in actual values after running data size sweep in notebook 02
a3_data = pd.DataFrame({
    'n_samples': [747, 1868, 3736, 7473],
    'accuracy': [None, None, None, None],
})

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(a3_data['n_samples'], a3_data['accuracy'], marker='o', color='mediumseagreen')
ax.set_xlabel('Training Samples')
ax.set_ylabel('Accuracy')
ax.set_title('A3 Ablation: Learning Curve')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot4_a3_data_size_ablation.png', dpi=150)
plt.show()

In [ ]:
summary_df.to_csv(RESULTS_DIR / 'summary_all_experiments.csv', index=False)
print('Saved: results/summary_all_experiments.csv')
summary_df[['experiment', 'accuracy', 'mean_latency_ms', 'mean_token_count', 'n_samples']]

## Exp 6 — Budget Router: Pareto Curve

Run cells below only after `results/exp6_router_full_sweep.csv` exists (generated by notebook 03 Exp 6 cells).

In [ ]:
# Plot 5: Budget router Pareto curve (accuracy vs mean tokens generated)
# Load sweep CSV from Exp 6
sweep_path = RESULTS_DIR / 'exp6_router_full_sweep.csv'

if not sweep_path.exists():
    print('[SKIP] exp6_router_full_sweep.csv not found. Run Exp 6 cells in notebook 03 first.')
else:
    sweep_df = pd.read_csv(sweep_path)

    # Separate finite-threshold points from anchor points
    finite = sweep_df[sweep_df['threshold'].apply(lambda t: t not in [float('-inf'), float('inf')])]
    anchors = sweep_df[sweep_df['threshold'].apply(lambda t: t in [float('-inf'), float('inf')])]
    pareto = sweep_df[sweep_df['pareto_optimal'] == True]

    fig, ax = plt.subplots(figsize=(9, 6))

    # All sweep points (grey)
    ax.scatter(finite['mean_token_count'], finite['accuracy'],
               color='lightgrey', s=80, zorder=2, label='Threshold sweep')

    # Pareto-optimal points (highlighted)
    ax.scatter(pareto['mean_token_count'], pareto['accuracy'],
               color='steelblue', s=120, zorder=3, label='Pareto-optimal')

    # Connect Pareto points with a dashed line
    pareto_sorted = pareto.sort_values('mean_token_count')
    ax.plot(pareto_sorted['mean_token_count'], pareto_sorted['accuracy'],
            color='steelblue', linestyle='--', linewidth=1, zorder=2)

    # Anchor points (all-easy and all-hard)
    for _, row in anchors.iterrows():
        label = 'All-easy (budget=128)' if row['pct_easy'] > 0.99 else 'All-hard (budget=300)'
        marker = '<' if row['pct_easy'] > 0.99 else '>'
        ax.scatter(row['mean_token_count'], row['accuracy'],
                   marker=marker, s=180, zorder=4, label=label,
                   color='tomato' if 'easy' in label else 'darkorange')

    # Horizontal reference lines for Exp 0–4 (if summary_df is available)
    ref_exps = {
        'Exp 0: Zero-shot': 'dimgray',
        'Exp 4: Filtered D2': 'mediumseagreen',
    }
    for exp_name, color in ref_exps.items():
        row = summary_df[summary_df['experiment'] == exp_name]
        if not row.empty:
            acc = row.iloc[0]['accuracy']
            ax.axhline(acc, color=color, linestyle=':', linewidth=1.2,
                       label=f'{exp_name} ({acc:.2%})')

    ax.set_xlabel('Mean Tokens Generated per Sample (cost proxy)', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title('Exp 6: Budget Router — Pareto Frontier\n'
                 'Accuracy vs. Computational Cost', fontsize=12)
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'plot5_exp6_router_pareto.png', dpi=150)
    plt.show()

    # Print Pareto-optimal rows
    print('\nPareto-optimal operating points:')
    print(pareto_sorted[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))